# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [1]:
!docker info

Client:
 Version:    29.8.0
 Context:    desktop-linux
 Debug Mode: false
 Plugins:
  agent: Docker AI Agent Runner (Docker Inc.)
    Version:  v1.140.0
    Path:     /Users/ivan/.docker/cli-plugins/docker-agent
  ai: Docker AI Agent - Ask Gordon (Docker Inc.)
    Version:  v1.31.0
    Path:     /Users/ivan/.docker/cli-plugins/docker-ai
  buildx: Docker Buildx (Docker Inc.)
    Version:  v0.37.1
    Path:     /Users/ivan/.docker/cli-plugins/docker-buildx
  compose: Docker Compose (Docker Inc.)
    Version:  v5.5.1
    Path:     /Users/ivan/.docker/cli-plugins/docker-compose
  debug: Get a shell into any image or container (Docker Inc.)
    Version:  0.0.47
    Path:     /Users/ivan/.docker/cli-plugins/docker-debug
  desktop: Docker Desktop commands (Docker Inc.)
    Version:  v0.4.4
    Path:     /Users/ivan/.docker/cli-plugins/docker-desktop
  dhi: CLI for managing Docker Hardened Images (Docker Inc.)
    Version:  v0.0.7
    Path:     /Users/ivan/.docker/cli-plugins/docker-dhi
  exte

### What is a container?

Containers are like a simplified virtual machihne.

Containers are an abstraction at the app layer that packages code and dependencies together. Multiple containers can run on the same machine and share the OS kernel with other containers, each running as isolated processes in user space. Containers take up less space than VMs (container images are typically tens of MBs in size), can handle more applications and require fewer VMs and Operating systems.

### Why do we use containers?

Containers only contain the necessary softwares in the smalllest possible version, so it does not take up much space. And the version of the softwares could be controled and fixed, ensure reproducibility..
- Reproducibility: the exact same tool version and dependencies run on every machine, today and in five years, so results can be regenerated.
- Portability: "works on my machine" becomes "works on every machine with Docker/Apptainer" (laptop, HPC cluster, cloud).
- No dependency hell: tools needing conflicting versions of Java, Python or libraries each live in their own container.
- Easy installation: one `docker pull` instead of a manual install procedure.
- Isolation: the tool cannot mess up the host system.

### What is a docker image?

An image is the read-only template/blueprint a container is created from: a stack of filesystem layers (base OS, installed packages, the tool) plus metadata such as the default command and environment variables. Images are built from a Dockerfile and stored in registries (Docker Hub, quay.io, community.wave.seqera.io).

### Let's run our first docker image:

### Login to docker

In [ ]:
# This you need to do on the command line directly
docker login

### Run your first docker container

In [7]:
!docker run hello-world


Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (amd64)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/



### Find the container ID

In [8]:
!docker ps -a
# only the hello-world containers, IDs only:
!docker ps -a -q --filter ancestor=hello-world

CONTAINER ID   IMAGE                                                         COMMAND                  CREATED         STATUS                     PORTS     NAMES
9262f336b733   hello-world                                                   "/hello"                 4 seconds ago   Exited (0) 3 seconds ago             exciting_ride
ca490cfdb6ad   community.wave.seqera.io/library/gzip_wget:041a8ba200c3fa3f   "/bin/bash -c 'eval …"   45 hours ago    Exited (8) 45 hours ago              nxf-6py2fDwEdql3uQQqLiLbraT1
9262f336b733


### Delete the container again, give prove its deleted

In [9]:
!docker rm 9262f336b733

9262f336b733


In [10]:
# proof: the container no longer appears in the list (the IMAGE is still there, see docker images)
!docker ps -a
!docker ps -a --filter ancestor=hello-world
!docker images hello-world

CONTAINER ID   IMAGE                                                         COMMAND                  CREATED        STATUS                    PORTS     NAMES
ca490cfdb6ad   community.wave.seqera.io/library/gzip_wget:041a8ba200c3fa3f   "/bin/bash -c 'eval …"   45 hours ago   Exited (8) 45 hours ago             nxf-6py2fDwEdql3uQQqLiLbraT1
CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES
                                                            i Info →   U  In Use
IMAGE                ID             DISK USAGE   CONTENT SIZE   EXTRA
hello-world:latest   5e2309035332       25.9kB         9.49kB        


### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

Please describe the steps you took to download and run the software for the example fastq file from last week below:

All in terminal:
1. Went to the FastQC website and downloaded the macOS version (`FastQC_v0.12.1.dmg`). Opening the app was blocked by macOS Gatekeeper ("Apple could not verify FastQC is free of malware") because the app is not signed by Apple. It could be allowed via System Settings → Privacy & Security → "Open Anyway", but this only gives the graphical interface, not the command-line tool the exercise asks for.
2. Switched to the platform-independent zip version and downloaded it in the terminal (macOS has no `wget`, so I used `curl`):
   `curl -L -O https://www.bioinformatics.babraham.ac.uk/projects/fastqc/fastqc_v0.12.1.zip`
3. check java version
4. Unzipped the archive: `unzip fastqc_v0.12.1.zip` (creates a `FastQC/` folder).
5. Made the wrapper script executable: `chmod +x FastQC/fastqc`.
6. Checked that it runs: `./FastQC/fastqc --version` → `FastQC v0.12.1`.
7. Created an output folder and ran it on last week's file:
    ./FastQC/fastqc day_02/fetchngs_out/fastq/SRX19144486_SRR23195516_1.fastq.gz -o fastqc_manual
8. Opened `open fastqc_manual/SRX19144486_SRR23195516_1_fastqc.html` 




### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

In [15]:
# pull the container
IMAGE = "community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae"
!docker pull {IMAGE}

0.12.1--9971ea336a9eddae: Pulling from library/fastqc

Digest: sha256:f97f7658a191bebe2f2a007cab5b815d380f89a948d6eb1481b046ab7618bebc[1A
Status: Downloaded newer image for community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae
community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae


# run the container and save the results to a new "fastqc_results" directory


In [17]:
%%bash
IMAGE="community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae"
mkdir -p fastqc_results
docker run --rm -u $(id -u):$(id -g) -v "$PWD/..":/data -w /data "$IMAGE" \
  fastqc day_02/fetchngs_out/fastq/SRX19144486_SRR23195516_1.fastq.gz -o day_04/fastqc_results
ls fastqc_results

application/gzip


Started analysis of SRX19144486_SRR23195516_1.fastq.gz
Approx 5% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 10% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 15% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 20% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 25% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 30% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 35% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 40% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 45% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 50% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 55% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 60% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 65% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 70% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 75% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 80% complete for SRX19144486_SRR23195516_1.fastq.gz
Ap

Analysis complete for SRX19144486_SRR23195516_1.fastq.gz



Fontconfig error: No writable cache directories
	/opt/conda/var/cache/fontconfig
	/.cache/fontconfig
	/.fontconfig


SRX19144486_SRR23195516_1_fastqc.html
SRX19144486_SRR23195516_1_fastqc.zip


### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

The first manual install may feel similar in effort, but the container approach is easier from now on: no Java to install, no PATH or permission problems, and the same single command works on every computer, the cluster and inside Nextflow. Switching versions just means changing the image tag. The manual install has to be repeated (and debugged) on every new machine.

### What would you say, which approach is more reproducible?

The container. The image pins the exact FastQC version and its whole environment (Java version, libraries, OS). A manual install depends on whatever Java and system happen to be on the machine, and "download the latest version" changes over time. Referencing the image by a fixed tag (or digest) lets anyone rerun the identical software later.

### Compare the file to last weeks fastqc results, are they identical?
### Is the fastqc version identical?

In [19]:
!find ../day_02 -name "*fastqc.zip"

../day_02/results/rnaseq/fastqc/trim/SNI_Oxy_SRR23195516_trimmed_2_val_2_fastqc.zip
../day_02/results/rnaseq/fastqc/trim/SNI_Oxy_SRR23195516_trimmed_1_val_1_fastqc.zip
../day_02/results/rnaseq/fastqc/trim/Sham_Oxy_SRR23195511_trimmed_2_val_2_fastqc.zip
../day_02/results/rnaseq/fastqc/trim/Sham_Oxy_SRR23195511_trimmed_1_val_1_fastqc.zip
../day_02/results/rnaseq/fastqc/raw/SNI_Oxy_SRR23195516_raw_1_fastqc.zip
../day_02/results/rnaseq/fastqc/raw/Sham_Oxy_SRR23195511_raw_1_fastqc.zip
../day_02/results/rnaseq/fastqc/raw/Sham_Oxy_SRR23195511_raw_2_fastqc.zip
../day_02/results/rnaseq/fastqc/raw/SNI_Oxy_SRR23195516_raw_2_fastqc.zip
../day_02/work/0f/9a7e2ad77111698c1546b6b9f40e31/Sham_Oxy_SRR23195511_raw_1_fastqc.zip
../day_02/work/0f/9a7e2ad77111698c1546b6b9f40e31/Sham_Oxy_SRR23195511_raw_2_fastqc.zip
../day_02/work/dd/0e9beaf643f8f90d234b97b8e95ee2/SNI_Oxy_SRR23195516_trimmed_2_val_2_fastqc.zip
../day_02/work/dd/0e9beaf643f8f90d234b97b8e95ee2/SNI_Oxy_SRR23195516_trimmed_1_val_1_fastqc.zip
..

In [20]:
%%bash
DOCKER=fastqc_results/SRX19144486_SRR23195516_1_fastqc.zip
MANUAL=../fastqc_manual/SRX19144486_SRR23195516_1_fastqc.zip
LAST=../day_02/results/rnaseq/fastqc/raw/SNI_Oxy_SRR23195516_raw_1_fastqc.zip

# version used by each run (first line of fastqc_data.txt)
echo "Docker:    $(unzip -p "$DOCKER" '*/fastqc_data.txt' | head -1)"
echo "Manual:    $(unzip -p "$MANUAL" '*/fastqc_data.txt' | head -1)"
echo "Last week: $(unzip -p "$LAST"   '*/fastqc_data.txt' | head -1)"

# compare the actual QC numbers
echo "--- Docker vs manual ---"
diff <(unzip -p "$DOCKER" '*/fastqc_data.txt') <(unzip -p "$MANUAL" '*/fastqc_data.txt') && echo "data identical"
echo "--- Docker vs last week ---"
diff <(unzip -p "$DOCKER" '*/fastqc_data.txt') <(unzip -p "$LAST" '*/fastqc_data.txt') && echo "data identical"

# the HTML files themselves
shasum fastqc_results/*_1_fastqc.html ../fastqc_manual/SRX19144486_SRR23195516_1_fastqc.html

Docker:    ##FastQC	0.12.1
Manual:    ##FastQC	0.12.1
Last week: ##FastQC	0.12.1
--- Docker vs manual ---
data identical
--- Docker vs last week ---
4c4
< Filename	SRX19144486_SRR23195516_1.fastq.gz
---
> Filename	SNI_Oxy_SRR23195516_raw_1.gz
e7c6bb9803f5c72ed2eae5273e6d8ce400df7160  fastqc_results/SRX19144486_SRR23195516_1_fastqc.html
2e779a4cb5bca13fa68c60758a7ca439f3e1236c  ../fastqc_manual/SRX19144486_SRR23195516_1_fastqc.html


Yes fastQc version identical.

## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [22]:

%%writefile my_dockerfile
# open the file "my_dockerfile" in a text editor (here: written directly from the notebook)
FROM ubuntu:22.04

# no interactive questions during apt-get install
ENV DEBIAN_FRONTEND=noninteractive

# install cowsay and clean the package cache to keep the image small
RUN apt-get update && \
    apt-get install -y --no-install-recommends cowsay && \
    rm -rf /var/lib/apt/lists/*

# on Debian/Ubuntu cowsay is installed into /usr/games, which is not on the default PATH
ENV PATH="/usr/games:${PATH}"

# default program and default argument
ENTRYPOINT ["cowsay"]
CMD ["Moo! Hello from inside a container"]

Overwriting my_dockerfile


### Explain the RUN and ENV lines you added to the file

- `FROM ubuntu:22.04`: the base image, a minimal Ubuntu Linux; everything else is built on top of it.
- `RUN apt-get update && apt-get install -y --no-install-recommends cowsay && rm -rf /var/lib/apt/lists/*`: `RUN` executes a shell command **at build time** and saves the result as a new image layer. `apt-get update` downloads the package lists (the base image ships without them), `install -y` installs cowsay without asking for confirmation, `--no-install-recommends` skips optional extras, and deleting `/var/lib/apt/lists` removes the downloaded lists so the image stays small. The commands are chained with `&&` in one `RUN` so they form a single layer and the update is never cached separately from the install.
- `ENV DEBIAN_FRONTEND=noninteractive`: sets an environment variable so apt never stops to ask interactive questions (there is no keyboard during a build).
- `ENV PATH="/usr/games:${PATH}"`: Ubuntu puts cowsay in `/usr/games`, which is not on the default `PATH`, so without this line `cowsay` gives "command not found". `ENV` variables persist into every container started from the image (unlike `export` inside a `RUN`).
- `ENTRYPOINT`/`CMD`: the program run when the container starts and its default argument (overridable on `docker run`).

In [24]:
!docker build -t my-cowsay -f my_dockerfile .



[+] Building 0.0s (0/1)                                    docker:desktop-linux
[+] Building 0.2s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from my_dockerfile                    0.0s
 => => transferring dockerfile: 401B                                       0.0s
 => [internal] load metadata for docker.io/library/ubuntu:22.04            0.1s
[+] Building 0.3s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from my_dockerfile                    0.0s
 => => transferring dockerfile: 401B                                       0.0s
 => [internal] load metadata for docker.io/library/ubuntu:22.04            0.3s
[+] Building 0.5s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from my_dockerfile                    0.0s
 => => transferring dockerfile: 401B                                       0.0s
 => [internal] load metadata for docker

In [25]:
!docker images my-cowsay


                                                            i Info →   U  In Use
IMAGE              ID             DISK USAGE   CONTENT SIZE   EXTRA
my-cowsay:latest   98b6083b5690        194MB         43.5MB        


In [26]:
!docker run --rm my-cowsay  "Hello"

 _______
< Hello >
 -------
        \   ^__^
         \  (oo)\_______
            (__)\       )\/\
                ||----w |
                ||     ||


## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [28]:
%%writefile salmon_docker
# use the file "salmon_docker" in this directory to build a new docker image
FROM ubuntu:22.04

ENV DEBIAN_FRONTEND=noninteractive

# curl to download, ca-certificates so curl can verify GitHub's HTTPS certificate
RUN apt-get update && \
    apt-get install -y --no-install-recommends curl ca-certificates && \
    rm -rf /var/lib/apt/lists/*

# download the pre-compiled salmon release and unpack it to /opt/salmon
RUN curl -fsSL -o /tmp/salmon.tar.gz \
      https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz && \
    mkdir -p /opt/salmon && \
    tar -xzf /tmp/salmon.tar.gz -C /opt/salmon --strip-components=1 && \
    rm /tmp/salmon.tar.gz

# make the salmon binary callable from anywhere
ENV PATH="/opt/salmon/bin:${PATH}"

CMD ["salmon", "--version"]

Overwriting salmon_docker


In [29]:
# build the image
# --platform linux/amd64 is needed on Apple Silicon Macs because the binary is x86_64 (harmless elsewhere)
!docker build --platform linux/amd64 -t my-salmon:1.5.2 -f salmon_docker .





[+] Building 0.0s (0/1)                                    docker:desktop-linux
[+] Building 0.2s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from salmon_docker                    0.0s
 => => transferring dockerfile: 846B                                       0.0s
 => [internal] load metadata for docker.io/library/ubuntu:22.04            0.2s
[+] Building 0.3s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from salmon_docker                    0.0s
 => => transferring dockerfile: 846B                                       0.0s
 => [internal] load metadata for docker.io/library/ubuntu:22.04            0.3s
[+] Building 0.5s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from salmon_docker                    0.0s
 => => transferring dockerfile: 846B                                       0.0s
 => [internal] load metadata for doc

In [30]:
# run the docker image to give out the version of salmon
!docker run --rm --platform linux/amd64 my-salmon:1.5.2 salmon --version
# expected output: salmon 1.5.2

salmon 1.5.2


## Do you think bioinformaticians have to create a docker image every time they want to run a tool?

No. For almost every bioconda tool a ready-made image already exists, so normally you just pull it. Writing your own Dockerfile is only needed for unusual tools, custom combinations, or your own code.


Find the salmon docker image online and run it on your computer.
https://quay.io/repository/biocontainers/salmon?tab=tags
1.5.2--h84f40af_0

What is https://biocontainers.pro/ ?
BioContainers is a community project (built on top of Bioconda) that automatically builds and hosts Docker and Singularity/Apptainer images for thousands of bioinformatics tools, one image per tool version, mostly at `quay.io/biocontainers/<tool>:<version>--<build>`. nf-core modules have long used these images.

In [31]:
SALMON = "quay.io/biocontainers/salmon:1.5.2--h84f40af_0"
!docker pull {SALMON}
!docker run --rm {SALMON} salmon --version

1.5.2--h84f40af_0: Pulling from biocontainers/salmon

Digest: sha256:4ae09a47788f08317bd2f758ac4c8804c9e87d88caf500c449e18ac4794d0332[1A
Status: Downloaded newer image for quay.io/biocontainers/salmon:1.5.2--h84f40af_0
quay.io/biocontainers/salmon:1.5.2--h84f40af_0
salmon 1.5.2


## Are there other ways to create Docker (or Apptainer) images?

What is https://seqera.io/containers/ ?

Yes. Besides writing a Dockerfile yourself or pulling BioContainers, you can let a service build the image from a package list. "Seqera Containers" is a free service by Seqera (the company behind Nextflow) built on their Wave technology: you choose one or more packages from Bioconda/conda-forge or PyPI, choose Docker or Singularity/Apptainer and the CPU architecture, and it builds the image on demand and gives you a permanent public URI (`community.wave.seqera.io/library/...`). Its big advantage is that you can combine several tools in one image without writing a Dockerfile; nf-core increasingly uses it for its modules. Other options are building from conda environment files, `apptainer build` from a definition file or directly from a Docker image (`apptainer pull docker://...`).